In [1]:
import pytest
import torch

fixtures for setup --

sample input data representing a customer's purchase history

In [2]:
def sample_input():
    return {
        "customer_id": "0000f1c71aafe",
        "history": ["513512003", "535035001", "677930066"]
    }

catalog of article IDs used to verify predictions belong to this set

In [3]:
def catalog():
    return {
        "513512003", "535035001", "677930066", "771528003",
        "793856005", "783925002", "784347001", "791152003",
        "803118001", "805772003", "678942032", "820601001"
    }

fixture to load the trained model for testing

In [4]:
def model():
    from model.ipynb import LSTMModel 
    return LSTMModel.load("model_path_or_config")

tests --

tests that the model's prediction returns a list of exactly 7 article IDs of correct type

In [5]:
def test_prediction_length(model, sample_input):
    predictions = model.predict(sample_input)
    assert isinstance(predictions, list), "Predictions should be a list"
    assert len(predictions) == 7, "Predictions should contain exactly 7 items"
    assert all(isinstance(p, str) or isinstance(p, int) for p in predictions), "Predictions must be article IDs"


test that all predicted article ids excist in the catalog of valid articles

In [6]:
def test_predictions_in_catalog(model, sample_input, catalog):
    predictions = model.predict(sample_input)
    assert all(str(p) in catalog for p in predictions), "Predicted items not in catalog"

test that the model does not predict duplicate articles in the recommendation list

In [7]:
def test_predictions_no_duplicates(model, sample_input):
    predictions = model.predict(sample_input)
    assert len(set(predictions)) == len(predictions), "Duplicate predictions found"

test that the model can handle a cold start (no purchase history) and still returns valid predictions

In [8]:
def test_cold_start_prediction(model):
    new_customer_data = {
        'customer_id': 'new_customer',
        'history': []
    }
    predictions = model.predict(new_customer_data)
    assert isinstance(predictions, list)
    assert len(predictions) == 7, "Cold start predictions should return 7 items"
    assert all(isinstance(p, str) or isinstance(p, int) for p in predictions), "Cold start predictions should be valid IDs"

test that compares predicted articles against actual articles, measuring model accuracy

In [9]:
def test_prediction_vs_actual(model, X_test, Y_test, le, device, min_avg_overlap=0.3):
    model.eval()
    total_overlap = 0
    n = len(X_test)

    with torch.no_grad():
        for i in range(n):
            x = torch.tensor(X_test[i], dtype=torch.long).unsqueeze(0).to(device)  # Prepare input tensor
            outputs = model(x)  # Get model output logits for articles
            top_preds_idx = torch.topk(outputs, 7).indices.squeeze(0).cpu().numpy()  # Get indices of top 7 predictions
            
            # Convert predicted and actual indices back to original article IDs
            predicted_articles = set(le.inverse_transform(top_preds_idx))
            actual_articles = set(le.inverse_transform(Y_test[i]))

            # Calculate fraction of overlap between predicted and actual articles
            overlap = len(predicted_articles & actual_articles) / len(actual_articles)
            total_overlap += overlap

    avg_overlap = total_overlap / n
    assert avg_overlap >= min_avg_overlap, f"Average overlap {avg_overlap:.3f} is below minimum threshold {min_avg_overlap}"